In [2]:
pip install notebook confluent-kafka pandas


   ---------------------------------------- 4/4 [isoduration]

Note: you may need to restart the kernel to use updated packages.


In [8]:
from confluent_kafka.admin import AdminClient, NewTopic

BOOTSTRAP = "localhost:9092,localhost:9094,localhost:9096"
admin = AdminClient({"bootstrap.servers": BOOTSTRAP})

topics = [
    ("urbanpulse.bus_gps", 12, 86_400_000),           # 24h retention
    ("urbanpulse.traffic_signals", 8, 604_800_000),    # 7d retention
    ("urbanpulse.air_quality", 4, 7_776_000_000),      # 90d retention
    ("urbanpulse.smart_meters", 10, 31_536_000_000),   # 365d retention
    ("urbanpulse.incidents", 6, 604_800_000),
    ("urbanpulse.ward_energy_summary", 6, 15_552_000_000),
    ("urbanpulse.health_advisories", 4, 7_776_000_000),
    ("urbanpulse.dlq", 3, 1_209_600_000),
]

new_topics = [
    NewTopic(name, num_partitions=p, replication_factor=3, config={"retention.ms": str(r)})
    for name, p, r in topics
]

futures = admin.create_topics(new_topics)
for topic, f in futures.items():
    try:
        f.result()
        print(f"Created {topic}")
    except Exception as e:
        print(f"{topic}: {e}")

Created urbanpulse.bus_gps
Created urbanpulse.traffic_signals
Created urbanpulse.air_quality
Created urbanpulse.smart_meters
Created urbanpulse.incidents
Created urbanpulse.ward_energy_summary
Created urbanpulse.health_advisories
Created urbanpulse.dlq


In [9]:
metadata = admin.list_topics(timeout=10)
for t in sorted(metadata.topics):
    if t.startswith("urbanpulse"):
        print(t, "-", len(metadata.topics[t].partitions), "partitions")

urbanpulse.air_quality - 4 partitions
urbanpulse.bus_gps - 12 partitions
urbanpulse.dlq - 3 partitions
urbanpulse.health_advisories - 4 partitions
urbanpulse.incidents - 6 partitions
urbanpulse.smart_meters - 10 partitions
urbanpulse.traffic_signals - 8 partitions
urbanpulse.ward_energy_summary - 6 partitions


In [ ]:
docker compose down 
docker compose up -d